# Basic Setup

In [1]:
import json
from pathlib import Path
from pprint import pprint
import re

import ollama
import tiktoken
from neo4j import GraphDatabase

from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs.graph_document import GraphDocument

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

# File Analysis

In [3]:
FILE_PATH = Path("churchill.txt")
text = FILE_PATH.read_text(encoding="utf-8")
# pprint(text[:1000])

In [4]:
file_name = FILE_PATH.name
num_chars = len(text)

words = text.split()
num_words = len(words)

sentences = re.split(r"(?<=[.!?])\s+", text.strip())
sentences = [s for s in sentences if s.strip()]
num_sentences = len(sentences)

In [5]:
print(f"File: {file_name}")
print(f"Characters: {num_chars:,}")
print(f"Words: {num_words:,}")
print(f"Sentences: {num_sentences:,}")

File: churchill.txt
Characters: 19,244
Words: 3,164
Sentences: 151


# Chunking

In [6]:
encoding_name="cl100k_base"
chunk_size = 300
chunk_overlap = 0
separators = ["\n\n", "\n", ". ", "! ", "? ", " ", ""]

In [7]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=encoding_name,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=separators,
)

In [8]:
chunks = text_splitter.split_text(text)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 17


In [9]:
tokenizer = tiktoken.get_encoding(encoding_name)

chunk_token_lengths = [
    len(tokenizer.encode(chunk))
    for chunk in chunks
]

print(f"Number of chunks: {len(chunks):,}")
print(f"Min tokens     : {min(chunk_token_lengths):,}")
print(f"Max tokens     : {max(chunk_token_lengths):,}")
print(f"Avg tokens     : {sum(chunk_token_lengths) / len(chunk_token_lengths):.2f}")

Number of chunks: 17
Min tokens     : 177
Max tokens     : 296
Avg tokens     : 236.12


In [10]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

# GraphDB Connection

In [11]:
NEO4J_URI = "bolt://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "graphrag_builder"

DB_NAME = "graphragdb"

In [12]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [13]:
with driver.session(database="system") as session:
    session.run(
        f"CREATE DATABASE {DB_NAME} IF NOT EXISTS"
    )

In [14]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    database=DB_NAME,
    refresh_schema=False,
)

# Loading Chunks

In [15]:
source_file = FILE_PATH.name
document_id = f"doc:{source_file.split('.')[0]}"

In [16]:
with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [17]:
chunk_records = [
    {
        "id": f"{document_id}:chunk:{i:04d}",
        "document_id": document_id,
        "text": chunk,
        "chunk_index": i,
        "token_count": chunk_token_lengths[i],
    }
    for i, chunk in enumerate(chunks)
]

print(f"Prepared {len(chunk_records)} chunks")

Prepared 17 chunks


# Injecting Document & Chunks 

In [18]:
graph.query("""
    CREATE CONSTRAINT document_id_unique IF NOT EXISTS
    FOR (d:Document)
    REQUIRE d.id IS UNIQUE
""")

[]

In [19]:
graph.query("""
    CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS
    FOR (c:Chunk)
    REQUIRE c.id IS UNIQUE
""")

[]

In [20]:
graph.query(
    """
    MERGE (d:Document {id: $document_id})
    SET
        d.file_name = $file_name,
        d.source = $source,
        d.file_type = "txt"

    WITH d

    UNWIND $chunks AS chunk

    MERGE (c:Chunk {id: chunk.id})
    SET
        c.document_id = chunk.document_id,
        c.text = chunk.text,
        c.chunk_index = chunk.chunk_index,
        c.token_count = chunk.token_count,
        c.source = $source

    MERGE (d)-[:HAS_CHUNK]->(c)
    """,
    params={
        "document_id": document_id,
        "file_name": source_file,
        "source": source_file,
        "chunks": chunk_records,
    },
)

print(f"Document '{source_file}' and {len(chunk_records)} chunks inserted")

Document 'churchill.txt' and 17 chunks inserted


# Building Relations

In [21]:
graph.query(
    """
    UNWIND range(0, size($chunks) - 2) AS i

    MATCH (c1:Chunk {id: $chunks[i].id})
    MATCH (c2:Chunk {id: $chunks[i + 1].id})

    MERGE (c1)-[:NEXT]->(c2)
    """,
    params={"chunks": chunk_records},
)

[]

In [22]:
result = graph.query("""
MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)
RETURN
    d.id AS document_id,
    d.file_name AS file_name,
    count(c) AS chunk_count
""")

for row in result:
    print(row)

{'document_id': 'doc:churchill', 'file_name': 'churchill.txt', 'chunk_count': 17}


In [23]:
result = graph.query("""
MATCH (c1:Chunk)-[:NEXT]->(c2:Chunk)
RETURN
    c1.chunk_index AS current_chunk,
    c2.chunk_index AS next_chunk
ORDER BY current_chunk
LIMIT 10
""")

for row in result:
    print(row)

{'current_chunk': 0, 'next_chunk': 1}
{'current_chunk': 1, 'next_chunk': 2}
{'current_chunk': 2, 'next_chunk': 3}
{'current_chunk': 3, 'next_chunk': 4}
{'current_chunk': 4, 'next_chunk': 5}
{'current_chunk': 5, 'next_chunk': 6}
{'current_chunk': 6, 'next_chunk': 7}
{'current_chunk': 7, 'next_chunk': 8}
{'current_chunk': 8, 'next_chunk': 9}
{'current_chunk': 9, 'next_chunk': 10}


# Adding Embedding

In [24]:
EMBEDDING_MODEL = "nomic-embed-text:137m-v1.5-fp16"
BATCH_SIZE = 10

In [25]:
for start in range(0, len(chunk_records), BATCH_SIZE):
    batch = chunk_records[start:start + BATCH_SIZE]

    texts = [item["text"] for item in batch]

    response = ollama.embed(
        model=EMBEDDING_MODEL,
        input=texts,
    )

    embeddings = response["embeddings"]

    graph.query(
        """
        UNWIND $items AS item

        MATCH (c:Chunk {id: item.id})

        SET c.embedding = item.embedding
        """,
        params={
            "items": [
                {
                    "id": item["id"],
                    "embedding": embedding,
                }
                for item, embedding in zip(batch, embeddings)
            ]
        },
    )

    print(
        f"Embedded and stored chunks "
        f"{start} → {start + len(batch) - 1}"
    )

Embedded and stored chunks 0 → 9
Embedded and stored chunks 10 → 16


In [26]:
result = graph.query("""
MATCH (c:Chunk)
RETURN
    count(c) AS total_chunks,
    count(c.embedding) AS embedded_chunks
""")

print(result)

[{'total_chunks': 17, 'embedded_chunks': 17}]


In [27]:
result = graph.query("""
MATCH (c:Chunk)
WHERE c.embedding IS NOT NULL
RETURN
    c.chunk_index AS chunk_index,
    size(c.embedding) AS dimensions
ORDER BY c.chunk_index
LIMIT 3
""")

for row in result:
    print(row)

{'chunk_index': 0, 'dimensions': 768}
{'chunk_index': 1, 'dimensions': 768}
{'chunk_index': 2, 'dimensions': 768}


In [28]:
graph.query("""
CREATE VECTOR INDEX chunk_embedding_index IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding
OPTIONS {
    indexConfig: {
        `vector.dimensions`: 768,
        `vector.similarity_function`: 'cosine'
    }
}
""")

print("Vector index created")

Vector index created


In [29]:
result = graph.query("""
SHOW VECTOR INDEXES
YIELD
    name,
    state,
    type,
    entityType,
    labelsOrTypes,
    properties,
    options
RETURN
    name,
    state,
    type,
    entityType,
    labelsOrTypes,
    properties,
    options
""")

In [30]:
result

[{'name': 'chunk_embedding_index',
  'state': 'ONLINE',
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['embedding'],
  'options': {'indexConfig': {'vector.dimensions': 768,
    'vector.default_search_expansion_factor': 1.5,
    'vector.hnsw.m': 16,
    'vector.quantization.type': 'SCALAR',
    'vector.similarity_function': 'COSINE',
    'vector.hnsw.ef_construction': 100}}}]

# Testing Vector Search

In [31]:
query = "What role did Winston Churchill play during World War II?"

query_embedding = ollama.embed(
    model=EMBEDDING_MODEL,
    input=query,
)["embeddings"][0]

In [32]:
result = graph.query(
    """
    MATCH (c:Chunk)
    SEARCH c IN (
        VECTOR INDEX chunk_embedding_index
        FOR $query_embedding
        LIMIT $top_k
    )
    SCORE AS score

    RETURN
        c.chunk_index AS chunk_index,
        c.token_count AS token_count,
        score,
        c.text AS text
    ORDER BY score DESC
    """,
    params={
        "top_k": 3,
        "query_embedding": query_embedding,
    },
)

for row in result:
    print(
        f"--- Chunk {row['chunk_index']} "
        f"| Score: {row['score']:.4f} ---"
    )
    # print(row["text"])

--- Chunk 4 | Score: 0.8593 ---
--- Chunk 3 | Score: 0.8592 ---
--- Chunk 9 | Score: 0.8555 ---


# Adding Neighbour Expanding with Vector Search

In [33]:
def vector_search(query: str, top_k: int = 3):
    query_embedding = ollama.embed(
        model=EMBEDDING_MODEL,
        input=query,
    )["embeddings"][0]

    results = graph.query(
        """
        MATCH (c:Chunk)
        SEARCH c IN (
            VECTOR INDEX chunk_embedding_index
            FOR $query_embedding
            LIMIT $top_k
        )
        SCORE AS score

        RETURN
            c.id AS chunk_id,
            c.chunk_index AS chunk_index,
            c.token_count AS token_count,
            score,
            c.text AS text
        ORDER BY score DESC
        """,
        params={
            "top_k": top_k,
            "query_embedding": query_embedding,
        },
    )

    return results

In [34]:
def expand_with_neighbors(results):
    chunk_ids = [row["chunk_id"] for row in results]

    expanded = graph.query(
        """
        UNWIND $chunk_ids AS chunk_id

        MATCH (c:Chunk {id: chunk_id})

        OPTIONAL MATCH (prev:Chunk)-[:NEXT]->(c)
        OPTIONAL MATCH (c)-[:NEXT]->(next:Chunk)

        WITH [prev, c, next] AS neighbors

        UNWIND neighbors AS neighbor

        WITH DISTINCT neighbor
        WHERE neighbor IS NOT NULL

        RETURN
            neighbor.id AS chunk_id,
            neighbor.chunk_index AS chunk_index,
            neighbor.token_count AS token_count,
            neighbor.text AS text

        ORDER BY chunk_index
        """,
        params={
            "chunk_ids": chunk_ids,
        },
    )

    return expanded

In [35]:
def retrieve(query: str, top_k: int = 3):
    # Step 1: retrieve the most similar chunks
    vector_results = vector_search(
        query=query,
        top_k=top_k,
    )

    # Step 2: expand the retrieved chunks with neighbors
    expanded_results = expand_with_neighbors(
        vector_results
    )

    return {
        "vector_results": vector_results,
        "expanded_results": expanded_results,
    }

In [39]:
query = "What role did Winston Churchill play during World War II?"

retrieval = retrieve(query, top_k=3)

print("TOP 3 VECTOR RESULTS")

for row in retrieval["vector_results"]:
    print(
        f"Chunk {row['chunk_index']} "
        f"| Score: {row['score']:.4f}"
    )

print("\nEXPANDED CONTEXT")

for row in retrieval["expanded_results"]:
    print(
        f"Chunk {row['chunk_index']} "
        f"| {row['token_count']} tokens"
    )
    # print(row["text"])

TOP 3 VECTOR RESULTS
Chunk 4 | Score: 0.8593
Chunk 3 | Score: 0.8592
Chunk 9 | Score: 0.8555

EXPANDED CONTEXT
Chunk 2 | 223 tokens
Chunk 3 | 296 tokens
Chunk 4 | 281 tokens
Chunk 5 | 208 tokens
Chunk 8 | 188 tokens
Chunk 9 | 274 tokens
Chunk 10 | 246 tokens


# Entities & Relations Extraction

In [40]:
model = "gpt-oss:120b-cloud"
temperature = 0

In [41]:
llm = ChatOllama(
    model=model,
    temperature=temperature,
    reasoning=False,
    validate_model_on_init=False,
)

In [42]:
def build_extraction_documents(
    chunks,
    document_id: str,
    chunks_per_window: int = 1,
):
    """
    Group consecutive chunks into extraction windows.

    Each window becomes one LangChain Document.
    The text is concatenated without chunk labels.
    Source chunk IDs are kept in metadata.
    """

    if chunks_per_window < 1:
        raise ValueError("chunks_per_window must be >= 1")

    documents = []

    for start in range(0, len(chunks), chunks_per_window):
        window = chunks[start:start + chunks_per_window]

        chunk_ids = [
            f"{document_id}:chunk:{i:04d}"
            for i in range(start, start + len(window))
        ]

        combined_text = "\n\n".join(window)

        documents.append(
            Document(
                page_content=combined_text,
                metadata={
                    "document_id": document_id,
                    "chunk_ids": chunk_ids,
                    "chunk_start": start,
                    "chunk_end": start + len(window) - 1,
                },
            )
        )

    return documents

In [43]:
extraction_docs = build_extraction_documents(
    chunks=chunks,
    document_id=document_id,
    chunks_per_window=3,
)

print(f"Number of extraction windows: {len(extraction_docs)}")

Number of extraction windows: 6


In [44]:
transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[],
    allowed_relationships=[],
    strict_mode=True,
    ignore_tool_usage=True
)

In [45]:
test_doc = extraction_docs[0]
test_doc.metadata

{'document_id': 'doc:churchill',
 'chunk_ids': ['doc:churchill:chunk:0000',
  'doc:churchill:chunk:0001',
  'doc:churchill:chunk:0002'],
 'chunk_start': 0,
 'chunk_end': 2}

In [46]:
graph_documents = transformer.convert_to_graph_documents(
    [test_doc]
)

In [47]:
graph_doc = graph_documents[0]

In [48]:
for i, node in enumerate(graph_doc.nodes[:10], start=1):
    print(f"{i}. {node.id} | {node.type}")

1. Egypt | Place
2. Leonard Jerome | Person
3. 7th Duke of Marlborough | Person
4. Dervishes | Group
5. John S Churchill | Person
6. Bolsheviks | Group
7. Germany | Place
8. Liberal Party | Organization
9. Lord Randolph Churchill | Person
10. Battle of Omdurman | Event


In [49]:
for i, rel in enumerate(graph_doc.relationships[:10], start=1):
    print(f"{i}. {rel.source.id} --[{rel.type}]--> {rel.target.id}")

1. Winston Churchill --[CHILD_OF]--> Lord Randolph Churchill
2. Winston Churchill --[CHILD_OF]--> Jennie Jerome
3. Lord Randolph Churchill --[CHILD_OF]--> 7th Duke of Marlborough
4. Jennie Jerome --[CHILD_OF]--> Leonard Jerome
5. Winston Churchill --[NURSED_BY]--> Mrs Everest
6. Winston Churchill --[SIBLING_OF]--> John S Churchill
7. Winston Churchill --[GRADUATED_FROM]--> Sandhurst
8. Winston Churchill --[SERVED_IN]--> 4th Hussars
9. Winston Churchill --[CAMPAIGN_IN]--> Cuba
10. Winston Churchill --[CAMPAIGN_IN]--> North West Frontier of India


In [50]:
CHUNKS_PER_WINDOW = 3

extraction_docs = build_extraction_documents(
    chunks=chunks,
    document_id=document_id,
    chunks_per_window=CHUNKS_PER_WINDOW,
)

print(f"Total chunks: {len(chunks)}")
print(f"Chunks per window: {CHUNKS_PER_WINDOW}")
print(f"Extraction windows: {len(extraction_docs)}")

Total chunks: 17
Chunks per window: 3
Extraction windows: 6


In [51]:
for i, doc in enumerate(extraction_docs):
    print(
        f"Window {i}: chunks {doc.metadata['chunk_start']} → {doc.metadata['chunk_end']}")

Window 0: chunks 0 → 2
Window 1: chunks 3 → 5
Window 2: chunks 6 → 8
Window 3: chunks 9 → 11
Window 4: chunks 12 → 14
Window 5: chunks 15 → 16


In [52]:
all_graph_documents = []

for i, extraction_doc in enumerate(extraction_docs):
    try:
        graph_doc = transformer.convert_to_graph_documents(
            [extraction_doc]
        )[0]

        graph_doc.source = extraction_doc
        all_graph_documents.append(graph_doc)

        print(
            f"Window {i + 1}/{len(extraction_docs)} "
            f"| Chunks {extraction_doc.metadata['chunk_start']}-"
            f"{extraction_doc.metadata['chunk_end']} "
            f"| Nodes: {len(graph_doc.nodes)} "
            f"| Edges: {len(graph_doc.relationships)}"
        )

    except Exception as e:
        print(f"Window {i + 1} ❌ {e}")

Window 1/6 | Chunks 0-2 | Nodes: 36 | Edges: 37
Window 2/6 | Chunks 3-5 | Nodes: 27 | Edges: 31
Window 3/6 | Chunks 6-8 | Nodes: 19 | Edges: 17
Window 4/6 | Chunks 9-11 | Nodes: 47 | Edges: 54
Window 5/6 | Chunks 12-14 | Nodes: 33 | Edges: 34
Window 6/6 | Chunks 15-16 | Nodes: 25 | Edges: 24


In [53]:
total_nodes = sum(
    len(doc.nodes)
    for doc in all_graph_documents
)

total_relationships = sum(
    len(doc.relationships)
    for doc in all_graph_documents
)

print(f"Extracted nodes: {total_nodes}")
print(f"Extracted relationships: {total_relationships}")

Extracted nodes: 187
Extracted relationships: 197


In [54]:
for graph_doc in all_graph_documents:
    metadata = graph_doc.source.metadata

    source_chunk_ids = metadata["chunk_ids"]

    for node in graph_doc.nodes:
        node.properties["source_document_id"] = metadata["document_id"]
        node.properties["source_chunk_ids"] = source_chunk_ids
        node.properties["extraction_chunk_start"] = metadata["chunk_start"]
        node.properties["extraction_chunk_end"] = metadata["chunk_end"]

    for rel in graph_doc.relationships:
        rel.properties["source_document_id"] = metadata["document_id"]
        rel.properties["source_chunk_ids"] = source_chunk_ids
        rel.properties["extraction_chunk_start"] = metadata["chunk_start"]
        rel.properties["extraction_chunk_end"] = metadata["chunk_end"]

In [55]:
def normalize_entity_key(node):
    return (
        node.type.strip().lower(),
        node.id.strip().lower(),
    )


def normalize_relation_key(rel):
    return (
        rel.source.type.strip().lower(),
        rel.source.id.strip().lower(),
        rel.type.strip().upper(),
        rel.target.type.strip().lower(),
        rel.target.id.strip().lower(),
    )

In [56]:
unique_nodes = {}

for graph_doc in all_graph_documents:
    for node in graph_doc.nodes:

        key = normalize_entity_key(node)

        if key not in unique_nodes:
            unique_nodes[key] = node

        else:
            existing = unique_nodes[key]

            existing_sources = set(
                existing.properties.get(
                    "source_chunk_ids", []
                )
            )

            new_sources = set(
                node.properties.get(
                    "source_chunk_ids", []
                )
            )

            existing.properties["source_chunk_ids"] = sorted(
                existing_sources | new_sources
            )

print(f"Unique nodes: {len(unique_nodes)}")

Unique nodes: 171


In [57]:
unique_relationships = {}

for graph_doc in all_graph_documents:
    for rel in graph_doc.relationships:

        key = normalize_relation_key(rel)

        if key not in unique_relationships:
            unique_relationships[key] = rel

        else:
            existing = unique_relationships[key]

            existing_sources = set(
                existing.properties.get(
                    "source_chunk_ids", []
                )
            )

            new_sources = set(
                rel.properties.get(
                    "source_chunk_ids", []
                )
            )

            existing.properties["source_chunk_ids"] = sorted(
                existing_sources | new_sources
            )

print(f"Unique relationships: {len(unique_relationships)}")

Unique relationships: 197


In [58]:
merged_graph_document = GraphDocument(
    nodes=list(unique_nodes.values()),
    relationships=list(unique_relationships.values()),
    source=extraction_docs[0],
)

In [59]:
for node in merged_graph_document.nodes:
    if node.type in {"Document", "Chunk"}:
        print(f"\n[{node.type}]")
        print(f"ID         : {node.id}")
        print(f"Properties : {node.properties}")

        relationships = []

        for rel in merged_graph_document.relationships:
            if rel.source.id == node.id:
                relationships.append(
                    f"  ({rel.source.type}) {rel.source.id}"
                    f" --[{rel.type}]--> "
                    f"({rel.target.type}) {rel.target.id}"
                )

            elif rel.target.id == node.id:
                relationships.append(
                    f"  ({rel.source.type}) {rel.source.id}"
                    f" --[{rel.type}]--> "
                    f"({rel.target.type}) {rel.target.id}"
                )

        if relationships:
            print("Relationships:")
            for relationship in relationships:
                print(relationship)
        else:
            print("Relationships: None")

In [60]:
graph.add_graph_documents(
    [merged_graph_document],
    include_source=False,
)

# Extraction Post-Processing

In [62]:
graph.query("""
    MATCH (e)
    WHERE e.source_chunk_ids IS NOT NULL
    SET e:__Entity__
""")

print("__Entity__ label added to all extracted entities")

__Entity__ label added to all extracted entities


In [63]:
graph.query("""
    MATCH (e:__Entity__)
    UNWIND e.source_chunk_ids AS chunk_id
    MATCH (c:Chunk {id: chunk_id})
    MERGE (c)-[:MENTIONS]->(e)
""")

print("MENTIONS relationships created")

MENTIONS relationships created


In [64]:
result = graph.query("""
    MATCH (c:Chunk)-[:MENTIONS]->(e:__Entity__)
    RETURN count(*) AS total_mentions
""")
print(result)

[{'total_mentions': 536}]


In [65]:
def get_entity_chunks(entity_name: str):
    result = graph.query(
        """
        MATCH (c:Chunk)-[:MENTIONS]->(e:__Entity__)
        WHERE toLower(e.id) = toLower($entity_name)

        RETURN
            c.chunk_index AS chunk_index,
            c.id AS chunk_id,
            c.token_count AS token_count,
            c.text AS text

        ORDER BY c.chunk_index
        """,
        params={
            "entity_name": entity_name,
        },
    )

    return result

In [71]:
entity = "Normandy"

results = get_entity_chunks(entity)

print(f"Entity: {entity}")
print(f"Chunks found: {len(results)}")

chunk_indices = [str(row["chunk_index"]) for row in results]

print(f"Chunks: {', '.join(chunk_indices)}")

combined_text = "\n\n".join(
    row["text"]
    for row in results
)

print("\nText:")
pprint(combined_text)

Entity: Normandy
Chunks found: 3
Chunks: 6, 7, 8

Text:
('However, America’s entry into the war also caused Churchill problems; as he '
 'said, the only thing worse than fighting a war with allies is fighting a war '
 'without them. At first, despite disasters such as the Japanese capture of '
 'Singapore early in 1942, Churchill was able to influence the Americans. He '
 'persuaded Roosevelt to fight Germany before Japan, and to follow the British '
 'strategy of trying to slit open the “soft underbelly” of Europe. This '
 'involved the invasions of North Africa, Sicily, and Italy – the last of '
 'which proved to have a very well armoured belly.\n'
 '\n'
 'It soon became apparent that Churchill was the littlest of the “Big Three”. '
 'At the Teheran Conference in November, 1943, he said, the “poor little '
 'English donkey” was squeezed between the great Russian bear and the mighty '
 'American buffalo, yet only he knew the way home.\n'
 '\n'
 'In June 1944 the Allies invaded Normand

# Vector Retrieval Tool

In [72]:
@tool
def document_retrieval(query: str) -> str:
    """
    Search the Churchill document using semantic vector retrieval.

    The tool retrieves the top 3 relevant chunks from Neo4j,
    expands them with previous and next chunks,
    removes duplicates, and returns the final context
    ordered according to the original document.
    """

    vector_results = vector_search(
        query=query,
        top_k=3,
    )

    if not vector_results:
        return "No relevant context was found."

    expanded_results = expand_with_neighbors(
        vector_results
    )

    if not expanded_results:
        return "No relevant context was found."

    context_parts = []

    for row in expanded_results:
        context_parts.append(
            f"[Chunk {row['chunk_index']}]\n"
            f"{row['text']}"
        )

    return "\n\n".join(context_parts)

In [74]:
result = document_retrieval.invoke({
    "query": "What role did Churchill play during World War II?"
})

print(result)

[Chunk 2]
Churchill rose swiftly within the Liberal ranks and became a Cabinet Minister in 1908 – President of the Board of Trade. In this capacity and as Home Secretary (1910-11) he helped to lay the foundations of the post-1945 welfare state.

His parliamentary career was far from being plain sailing and he made a number of spectacular blunders, so much so that he was often accused of having genius without judgement. The chief setback of his career occurred in 1915 when, as First Lord of the Admiralty, he sent a naval force to the Dardanelles in an attempt to knock Turkey out of the war and to outflank Germany on a continental scale. The expedition was a disaster and it marked the lowest point in Churchill’s fortunes.

However, Churchill could not be kept out of power for long and Lloyd George, anxious to draw on his talents and to spike his critical guns, soon re-appointed him to high office. Their relationship was not always a comfortable one, particularly when Churchill tried to i

# Entity Finder Tool

In [75]:
graph.query("""
    CREATE FULLTEXT INDEX entity_id_fulltext_index IF NOT EXISTS
    FOR (e:__Entity__)
    ON EACH [e.id]
""")

print("Entity full-text index created")

Entity full-text index created


In [77]:
result = graph.query("""
    SHOW FULLTEXT INDEXES
    YIELD name, state, entityType, labelsOrTypes, properties
    WHERE name = 'entity_id_fulltext_index'
    RETURN name, state, entityType, labelsOrTypes, properties
""")

result

[{'name': 'entity_id_fulltext_index',
  'state': 'ONLINE',
  'entityType': 'NODE',
  'labelsOrTypes': ['__Entity__'],
  'properties': ['id']}]

In [78]:
@tool
def find_entity(name: str) -> str:
    """
    Find entities in the Churchill knowledge graph by name.

    Uses Neo4j full-text search over the entity id property
    and returns the closest matching entities ordered by score.
    """

    if not name or not name.strip():
        return "No entity name was provided."

    results = graph.query(
        """
        CALL db.index.fulltext.queryNodes(
            'entity_id_fulltext_index',
            $query
        )
        YIELD node, score

        RETURN
            node.id AS entity_id,
            node.type AS entity_type,
            score
        ORDER BY score DESC
        LIMIT 5
        """,
        params={
            "query": name.strip(),
        },
    )

    if not results:
        return f"No entity matches found for: {name}"

    lines = []

    for i, row in enumerate(results, start=1):
        lines.append(
            f"{i}. "
            f"ID: {row['entity_id']} | "
            f"Type: {row['entity_type']} | "
            f"Score: {row['score']:.4f}"
        )

    return "\n".join(lines)

In [79]:
result = find_entity.invoke({
    "name": "Winston Churchill"
})

print(result)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 10, column: 18, offset: 206} for query: "\n        CALL db.index.fulltext.queryNodes(\n            'entity_id_fulltext_index',\n            $query\n        )\n        YIELD node, score\n\n        RETURN\n            node.id AS entity_id,\n            node.type AS entity_type,\n            score\n        ORDER BY score DESC\n        LIMIT 5\n        "


1. ID: Winston Churchill | Type: None | Score: 3.3711
2. ID: Winston | Type: None | Score: 2.4785
3. ID: Mary Churchill | Type: None | Score: 1.3698
4. ID: Diana Churchill | Type: None | Score: 1.3698
5. ID: Sarah Churchill | Type: None | Score: 1.3698


In [80]:
result = find_entity.invoke({
    "name": "Churchill"
})

print(result)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 10, column: 18, offset: 206} for query: "\n        CALL db.index.fulltext.queryNodes(\n            'entity_id_fulltext_index',\n            $query\n        )\n        YIELD node, score\n\n        RETURN\n            node.id AS entity_id,\n            node.type AS entity_type,\n            score\n        ORDER BY score DESC\n        LIMIT 5\n        "


1. ID: Sarah Churchill | Type: None | Score: 1.3698
2. ID: Clementine Churchill | Type: None | Score: 1.3698
3. ID: Marigold Churchill | Type: None | Score: 1.3698
4. ID: Mary Churchill | Type: None | Score: 1.3698
5. ID: Winston Churchill | Type: None | Score: 1.3698


# Entities Relationships Tool

In [81]:
@tool
def get_entity_relationships(
    entity_id: str,
    limit: int = 20,
    offset: int = 0,
) -> str:
    """
    Get direct semantic relationships for an entity.

    Returns relationships between entities only.
    MENTIONS relationships to chunks are excluded.

    Results are paginated using limit and offset.
    """

    if not entity_id or not entity_id.strip():
        return "No entity ID was provided."

    if limit <= 0:
        return "limit must be greater than 0."

    if offset < 0:
        return "offset must be >= 0."

    # Total count
    count_result = graph.query(
        """
        MATCH (e:__Entity__ {id: $entity_id})-[r]-(other:__Entity__)
        WHERE type(r) <> 'MENTIONS'
        RETURN count(r) AS total
        """,
        params={"entity_id": entity_id.strip()},
    )

    total = count_result[0]["total"] if count_result else 0

    if total == 0:
        return f"No semantic relationships found for entity: {entity_id}"

    # Paginated results
    results = graph.query(
        """
        MATCH (e:__Entity__ {id: $entity_id})-[r]-(other:__Entity__)
        WHERE type(r) <> 'MENTIONS'

        RETURN
            startNode(r).id AS start_entity,
            type(r) AS relationship,
            endNode(r).id AS end_entity,
            other.type AS other_entity_type

        ORDER BY relationship, end_entity
        SKIP $offset
        LIMIT $limit
        """,
        params={
            "entity_id": entity_id.strip(),
            "offset": offset,
            "limit": limit,
        },
    )

    if not results:
        return (
            f"No relationships found at offset {offset}. "
            f"Total relationships: {total}."
        )

    start = offset + 1
    end = offset + len(results)
    has_more = end < total

    lines = [
        f"Entity: {entity_id}",
        f"Total relationships: {total}",
        f"Showing: {start}-{end}",
        f"Has more: {has_more}",
        "",
    ]

    for i, row in enumerate(results, start=start):
        lines.append(
            f"{i}. "
            f"{row['start_entity']} "
            f"--[{row['relationship']}]--> "
            f"{row['end_entity']}"
        )

    if has_more:
        lines.append("")
        lines.append(
            f"To retrieve more, call this tool again "
            f"with offset={end}."
        )

    return "\n".join(lines)

In [82]:
result = get_entity_relationships.invoke({
    "entity_id": "Winston Churchill",
    "limit": 20,
    "offset": 20,
})

print(result)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 9, column: 19, offset: 263} for query: "\n        MATCH (e:__Entity__ {id: $entity_id})-[r]-(other:__Entity__)\n        WHERE type(r) <> 'MENTIONS'\n\n        RETURN\n            startNode(r).id AS start_entity,\n            type(r) AS relationship,\n            endNode(r).id AS end_entity,\n            other.type AS other_entity_type\n\n        ORDER BY relationship, end_entity\n        SKIP $offset\n        LIMIT $limit\n        "


Entity: Winston Churchill
Total relationships: 138
Showing: 21-40
Has more: True

21. Winston Churchill --[BLAMED_FOR_FAILURES]--> Norwegian Campaign
22. Winston Churchill --[BORN_ON]--> November 30, 1874
23. Winston Churchill --[BOUGHT]--> Chartwell
24. Winston Churchill --[BOUGHT]--> Colonist II
25. Winston Churchill --[BOUGHT]--> Farm adjoining Chartwell
26. Winston Churchill --[BROADCASTED_CALL_TO_DEFAT]--> Japan
27. Winston Churchill --[BROADCASTED_TO]--> nation
28. Winston Churchill --[BURIED_IN]--> Bladon churchyard
29. Winston Churchill --[BURIED_NEAR]--> Blenheim Palace
30. Winston Churchill --[CALLED_FOR_POLICY]--> several years of quiet steady administration
31. Winston Churchill --[CHILD_OF]--> Jennie Jerome
32. Winston Churchill --[CHILD_OF]--> Lord Randolph Churchill
33. Randolph Churchill --[CHILD_OF]--> Winston Churchill
34. Sarah Churchill --[CHILD_OF]--> Winston Churchill
35. Mary Churchill --[CHILD_OF]--> Winston Churchill
36. Winston Churchill --[COMMISSIONED_IN]-->

# Build Agent with Tools

In [83]:
tools = [
    find_entity,
    get_entity_relationships,
    document_retrieval,
]

In [84]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You answer questions about a document and its knowledge graph.

Choose tools based on what kind of information is needed.

TOOLS:

1. document_retrieval
Retrieves relevant passages from the source document using semantic search.
Use it when the answer depends on information contained in the document text,
such as facts, events, descriptions, dates, explanations, or details about a topic.

2. find_entity
Resolves a name to an entity in the knowledge graph.
Use it when you need to identify or disambiguate a specific person, place,
organization, event, or other graph entity.
This tool only identifies the entity; it does not provide its relationships.

3. get_entity_relationships
Traverses the knowledge graph from a specific entity and returns its direct
connections to other entities.
Use it when the main information needed is how entities are connected,
associated, related, connected, owned, joined, led, appointed, parented,
located, or otherwise linked.

For graph relationship questions:
- First resolve the relevant entity with find_entity.
- Then use its exact entity ID with get_entity_relationships.

General routing principle:
- Need information from the source text → document_retrieval
- Need to identify an entity → find_entity
- Need connections between entities → find_entity + get_entity_relationships

Do not use graph relationship tools merely because an entity is mentioned.
Do not use document_retrieval as a substitute for graph traversal when the
main requirement is understanding connections between entities.

Use multiple tools only when the question genuinely requires information
from more than one source.
"""
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [85]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

In [86]:
response = agent_executor.invoke({
    "input": "What role did Churchill play during World War II?"
})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `document_retrieval` with `{'query': 'Churchill role during World War II'}`


[Chunk 2]
Churchill rose swiftly within the Liberal ranks and became a Cabinet Minister in 1908 – President of the Board of Trade. In this capacity and as Home Secretary (1910-11) he helped to lay the foundations of the post-1945 welfare state.

His parliamentary career was far from being plain sailing and he made a number of spectacular blunders, so much so that he was often accused of having genius without judgement. The chief setback of his career occurred in 1915 when, as First Lord of the Admiralty, he sent a naval force to the Dardanelles in an attempt to knock Turkey out of the war and to outflank Germany on a continental scale. The expedition was a disaster and it marked the lowest point in Churchill’s fortunes.

However, Churchill could not be kept out of power for long and Lloyd George, anxious to draw on his talents and to spike his critical guns, 

In [92]:
response = agent_executor.invoke({
    "input": "Which places are directly associated with Churchill?"
})

print(response["output"])



> Entering new AgentExecutor chain...


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 10, column: 18, offset: 206} for query: "\n        CALL db.index.fulltext.queryNodes(\n            'entity_id_fulltext_index',\n            $query\n        )\n        YIELD node, score\n\n        RETURN\n            node.id AS entity_id,\n            node.type AS entity_type,\n            score\n        ORDER BY score DESC\n        LIMIT 5\n        "



Invoking: `find_entity` with `{'name': 'Churchill'}`


1. ID: Sarah Churchill | Type: None | Score: 1.3698
2. ID: Clementine Churchill | Type: None | Score: 1.3698
3. ID: Marigold Churchill | Type: None | Score: 1.3698
4. ID: Mary Churchill | Type: None | Score: 1.3698
5. ID: Winston Churchill | Type: None | Score: 1.3698

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 10, column: 18, offset: 206} for query: "\n        CALL db.index.fulltext.queryNodes(\n            'entity_id_fulltext_index',\n            $query\n        )\n        YIELD node, score\n\n        RETURN\n            node.id AS entity_id,\n            node.type AS entity_type,\n            score\n        ORDER BY score DESC\n        LIMIT 5\n        "



Invoking: `find_entity` with `{'name': 'Winston Churchill'}`


1. ID: Winston Churchill | Type: None | Score: 3.3711
2. ID: Winston | Type: None | Score: 2.4785
3. ID: Mary Churchill | Type: None | Score: 1.3698
4. ID: Diana Churchill | Type: None | Score: 1.3698
5. ID: Sarah Churchill | Type: None | Score: 1.3698

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 9, column: 19, offset: 263} for query: "\n        MATCH (e:__Entity__ {id: $entity_id})-[r]-(other:__Entity__)\n        WHERE type(r) <> 'MENTIONS'\n\n        RETURN\n            startNode(r).id AS start_entity,\n            type(r) AS relationship,\n            endNode(r).id AS end_entity,\n            other.type AS other_entity_type\n\n        ORDER BY relationship, end_entity\n        SKIP $offset\n        LIMIT $limit\n        "



Invoking: `get_entity_relationships` with `{'entity_id': 'Winston Churchill', 'limit': 100, 'offset': 0}`


Entity: Winston Churchill
Total relationships: 138
Showing: 1-100
Has more: True

1. Winston Churchill --[ACCUSED_OF]--> Labour leaders
2. Christopher Soames --[ADVISED]--> Winston Churchill
3. Winston Churchill --[AFTER_EVENT]--> Great War
4. Winston Churchill --[ANNOUNCED_START_OF]--> Cold War
5. Stanley Baldwin --[APPOINTED]--> Winston Churchill
6. Neville Chamberlain --[APPOINTED]--> Winston Churchill
7. Winston Churchill --[APPOINTED_AS]--> First Lord of the Admiralty
8. Winston Churchill --[APPOINTED_AS]--> Home Secretary
9. Winston Churchill --[APPOINTED_AS]--> President of the Board of Trade
10. Winston Churchill --[ASSIGNED_TO]--> Kitchener's army
11. Winston Churchill --[ATTEMPTED_CAMPAIGN]--> Dardanelles Campaign
12. Winston Churchill --[ATTENDED]--> Teheran Conference
13. Randolph Churchill --[AUTHORED_BIOGRAPHY_OF]--> Winston Churchill
14. Winston Churchill --[AUTHO

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: type)} {position: line: 9, column: 19, offset: 263} for query: "\n        MATCH (e:__Entity__ {id: $entity_id})-[r]-(other:__Entity__)\n        WHERE type(r) <> 'MENTIONS'\n\n        RETURN\n            startNode(r).id AS start_entity,\n            type(r) AS relationship,\n            endNode(r).id AS end_entity,\n            other.type AS other_entity_type\n\n        ORDER BY relationship, end_entity\n        SKIP $offset\n        LIMIT $limit\n        "



Invoking: `get_entity_relationships` with `{'entity_id': 'Winston Churchill', 'limit': 100, 'offset': 100}`


Entity: Winston Churchill
Total relationships: 138
Showing: 101-138
Has more: False

101. Winston Churchill --[PROPOSED_PARTNERSHIP_WITH]--> France
102. Winston Churchill --[PROPOSED_PARTNERSHIP_WITH]--> Germany
103. Winston Churchill --[PROPOSED_UNITED_STATES_OF_EUROPE]--> United States of Europe
104. Winston Churchill --[REACTED_WITH_JUBILATION_TO]--> Pearl Harbor attack
105. Winston Churchill --[REAPPOINTED_BY]--> David Lloyd George
106. Winston Churchill --[RECEIVED_HELP_FROM]--> United States
107. Winston Churchill --[RECEIVED_STATE_FUNERAL]--> state funeral
108. Winston Churchill --[RECEIVED_TELEGRAMS_FROM]--> World
109. Winston Churchill --[REIGN_PERIOD]--> twentieth century
110. Winston Churchill --[REMAINED_MEMBER_OF]--> Parliament
111. Winston Churchill --[REPRESENTED]--> Oldham
112. Winston Churchill --[RESIGNED_FROM_POSITION]--> Prime Minister
113. Winston Churchil